# N30 — RAG Agent

The RAG Agent answers one question: **what does the FIA regulation say about this?**

It builds a retrieval-augmented generation pipeline over the FIA Sporting Regulations and Technical Regulations (2023–2025). Given a natural-language query from the Strategy Orchestrator, it retrieves the most relevant regulation chunks from a local Qdrant vector store and returns them as structured `RegulationContext` objects — with article references and relevance scores.

This agent is event-driven: N31 activates it when N27 reports `sc_prob > 0.3` (safety car rules become relevant) or when N29 detects a `PROBLEM` or `ORDER` radio intent (driver or team directive that may have a regulatory dimension).

## Pipeline position

<pre>
query (natural language, from N31 Orchestrator)
    │
    ▼
N30 — RAG Agent  (LangGraph ReAct)
    │
    ├── query_rag_tool         dense retrieval from Qdrant + reranking
    │
    └── RegulationContext
            ├── chunks[]       relevant regulation text passages
            ├── articles[]     article/section references (e.g. Art. 48.3)
            └── scores[]       cosine similarity scores
</pre>

## Stack
- **Embedding model** — `all-MiniLM-L6-v2` (sentence-transformers, 384-dim, ~80ms/query CPU)
- **Vector store** — Qdrant local (on-disk, no Docker required)
- **Documents** — FIA Sporting Regulations + Technical Regulations 2023–2025 (PDF → text → chunks)
- **Chunking** — 512-token overlapping windows (overlap 64 tokens)
- **Retrieval** — cosine similarity top-k, k=5 per query

## Steps
- **Step 0** — Setup, imports & Qdrant connection
- **Step 1** — PDF ingestion + text extraction
- **Step 2** — Chunking + embedding + Qdrant indexing
- **Step 3** — `query_rag_tool` (@tool decorated retrieval function)
- **Step 4** — `RegulationContext` dataclass + `run_rag_agent(question)` entry point
- **Step 5** — LangGraph ReAct agent (`@tool` + `create_react_agent`)
- **Step 6** — Demo queries (SC rules, tyre regulations, pit lane procedures)
- **Step 7** — Export schema & config

---

In [ ]:
# ── Step 0: Setup & dependencies ────────────────────────────────────────────
# pip install qdrant-client sentence-transformers pymupdf langchain langchain-openai langgraph

import sys
import json
from pathlib import Path
from dataclasses import dataclass, field

repo_root = Path.cwd()
while not (repo_root / '.git').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

# ── RAG stack ────────────────────────────────────────────────────────────────
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
import fitz  # PyMuPDF — PDF text extraction

# ── Paths ────────────────────────────────────────────────────────────────────
RAG_DIR      = repo_root / 'data' / 'rag'
DOCS_DIR     = RAG_DIR / 'documents'       # FIA PDFs go here
QDRANT_PATH  = RAG_DIR / 'qdrant_local'    # on-disk Qdrant storage
OUTPUTS_DIR  = repo_root / 'notebooks' / 'agents' / 'outputs'

for d in [DOCS_DIR, QDRANT_PATH, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Qdrant + embedding config ────────────────────────────────────────────────
COLLECTION_NAME = 'fia_regulations'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
EMBEDDING_DIM   = 384
CHUNK_SIZE      = 512    # characters (not tokens — simpler, good enough for these docs)
CHUNK_OVERLAP   = 64
TOP_K           = 5      # chunks returned per query

# ── Load embedding model + Qdrant client ────────────────────────────────────
encoder = SentenceTransformer(EMBEDDING_MODEL)
client  = QdrantClient(path=str(QDRANT_PATH))

print(f"Embedding model : {EMBEDDING_MODEL}  (dim={EMBEDDING_DIM})")
print(f"Qdrant storage  : {QDRANT_PATH.relative_to(repo_root)}")
print(f"Documents dir   : {DOCS_DIR.relative_to(repo_root)}")
print(f"PDFs found      : {len(list(DOCS_DIR.glob('*.pdf')))}")

# Check if collection already exists
existing = [c.name for c in client.get_collections().collections]
print(f"Qdrant collections: {existing if existing else '(none yet)'}")